# Assistente de Perguntas e Respostas sobre Políticas Internas

## P0 — Preparação e leitura do corpus

In [1]:
# P0 - Leia os metadados e carregue o texto completo de cada documento.
from pathlib import Path
import pandas as pd


def carregar_corpus(base_dir):
    """Carrega os metadados e o conteúdo completo dos 12 documentos."""
    documentos = pd.read_csv(base_dir / "metadados.csv")
    documentos["texto"] = ""

    # Mantém o corpus associado ao arquivo indicado em cada linha dos metadados.
    for index, row in documentos.iterrows():
        caminho_arquivo = base_dir / "corpus" / row["arquivo"]
        documentos.at[index, "texto"] = caminho_arquivo.read_text(encoding="utf-8")

    return documentos


base_dir = Path.cwd()
df = carregar_corpus(base_dir)
status_counts = df["status"].value_counts()

assert len(df) == 12
assert status_counts["vigente"] == 11
assert status_counts["revogada"] == 1

display(status_counts)
display(df)

status
vigente     11
revogada     1
Name: count, dtype: int64

,doc_id,titulo,area_responsavel,vigencia_inicio,versao,status,arquivo,texto
0,POL-001,Política de Onboarding,Pessoas e Cultura,2026-02-01,2.0,vigente,POL-001_onboarding.md,# POL-001 Política de Onboarding\nEmpresa: Hor...
1,POL-002,Política de Férias,Pessoas e Cultura,2026-01-01,3.1,vigente,POL-002_ferias.md,# POL-002 Política de Férias\nEmpresa: Horizon...
2,POL-003,Política de Benefícios,Pessoas e Cultura,2026-01-01,4.0,vigente,POL-003_beneficios.md,# POL-003 Política de Benefícios\nEmpresa: Hor...
3,POL-004,Política de Home Office (versão 1),Pessoas e Cultura,2024-03-01,1.0,revogada,POL-004_home_office_v1_REVOGADA.md,# POL-004 Política de Home Office (versão 1)\n...
4,POL-005,Política de Trabalho Híbrido (versão 2),Pessoas e Cultura,2026-01-01,2.0,vigente,POL-005_trabalho_hibrido_v2.md,# POL-005 Política de Trabalho Híbrido (versão...
5,POL-006,Política de Reembolso de Despesas,Financeiro,2026-01-01,2.2,vigente,POL-006_reembolso.md,# POL-006 Política de Reembolso de Despesas\nE...
6,POL-007,Política de Avaliação de Desempenho,Pessoas e Cultura,2026-01-01,1.3,vigente,POL-007_avaliacao_desempenho.md,# POL-007 Política de Avaliação de Desempenho\...
7,POL-008,Política de Desligamento,Pessoas e Cultura,2026-01-01,1.1,vigente,POL-008_desligamento.md,# POL-008 Política de Desligamento\nEmpresa: H...
8,POL-009,Política de Segurança da Informação,Segurança da Informação,2026-01-01,3.0,vigente,POL-009_seguranca_informacao.md,# POL-009 Política de Segurança da Informação\...
9,POL-010,Política de Treinamento e Desenvolvimento,Pessoas e Cultura,2026-01-01,2.0,vigente,POL-010_treinamento.md,# POL-010 Política de Treinamento e Desenvolvi...


## Evidência da P0

In [2]:
# Evidência da P0
import re

# Conta palavras no texto completo de cada documento, incluindo o cabeçalho.
df["numero_palavras"] = df["texto"].apply(
    lambda texto: len(re.findall(r"[^\W_]+(?:[-'][^\W_]+)*", texto, flags=re.UNICODE))
)

# Tabela com doc_id, status e número de palavras de cada documento
resultado = df[["doc_id", "status", "numero_palavras"]].copy()
resultado = resultado.rename(columns={"numero_palavras": "número de palavras"})

resultado

,doc_id,status,número de palavras
0,POL-001,vigente,159
1,POL-002,vigente,130
2,POL-003,vigente,122
3,POL-004,revogada,87
4,POL-005,vigente,114
5,POL-006,vigente,111
6,POL-007,vigente,108
7,POL-008,vigente,113
8,POL-009,vigente,136
9,POL-010,vigente,87


---

## P1 - Chunking por seção

In [3]:
# P1 - Chunking por seção
import re


def criar_chunks(documentos):
    """Divide cada documento em um chunk por seção Markdown marcada por ##."""
    chunks = []

    for _, row in documentos.iterrows():
        linhas = row["texto"].splitlines()
        nome_secao_atual = None
        texto_secao = []

        for linha in linhas:
            # O cabeçalho Empresa: contém contexto do documento, não uma seção.
            if linha.startswith("Empresa:"):
                continue

            if re.match(r"^##\s+", linha):
                if nome_secao_atual is not None:
                    chunks.append({
                        "doc_id": row["doc_id"],
                        "titulo do documento": row["titulo"],
                        "nome da seção": nome_secao_atual,
                        "status": row["status"],
                        "texto da seção": " ".join(texto_secao).strip(),
                    })

                nome_secao_atual = re.sub(r"^##\s*", "", linha).strip()
                texto_secao = []
            elif nome_secao_atual is not None:
                texto_secao.append(linha.rstrip())

        if nome_secao_atual is not None:
            chunks.append({
                "doc_id": row["doc_id"],
                "titulo do documento": row["titulo"],
                "nome da seção": nome_secao_atual,
                "status": row["status"],
                "texto da seção": " ".join(texto_secao).strip(),
            })

    return chunks


# Cada seção ## recebe seus metadados e pode ser transformada em uma linha do DataFrame.
chunks = criar_chunks(df)
chunks_df = pd.DataFrame(chunks)
quantidade_total = len(chunks_df)
quantidade_por_documento = chunks_df["doc_id"].value_counts().sort_index()

print(f"Total de chunks: {quantidade_total}")
print("\nChunks por documento:")
print(quantidade_por_documento)

chunks_df

# Exibição opcional: descomente somente uma das opções abaixo.


# Exibição completa com o texto da seção sem truncamento:
pd.set_option("display.max_colwidth", None)
display(chunks_df)


# Exibição unitária a partir do índice, com todos os campos completos:

# indice_chunk = 0
# chunk_unitario = chunks_df.iloc[indice_chunk]
# for coluna, valor in chunk_unitario.items():
#    print(f"\n{coluna}: {valor}")


Total de chunks: 46

Chunks por documento:
doc_id
FAQ-001    6
POL-001    3
POL-002    4
POL-003    4
POL-004    3
POL-005    3
POL-006    4
POL-007    4
POL-008    4
POL-009    4
POL-010    3
POL-011    4
Name: count, dtype: int64


,doc_id,titulo do documento,nome da seção,status,texto da seção
0,POL-001,Política de Onboarding,Objetivo,vigente,Esta política define as etapas dos primeiros 30 dias de um novo colaborador na Horizonte Tech.
1,POL-001,Política de Onboarding,Primeira semana,vigente,Todo novo colaborador recebe um buddy. O buddy é um colega da mesma área que acompanha a integração nas primeiras 4 semanas. Os acessos a sistemas são liberados em até 2 dias úteis após a admissão. O pedido de acessos é feito pelo gestor direto no portal de TI. O treinamento de segurança da informação é obrigatório e deve ser concluído na primeira semana.
2,POL-001,Política de Onboarding,Primeiro mês,vigente,"Ao final de 30 dias, o gestor realiza uma conversa de alinhamento com o colaborador. Essa conversa não gera nota. O objetivo é revisar expectativas e o plano de trabalho. O checklist de onboarding fica disponível no portal de Pessoas e Cultura e deve ser concluído até o dia 30."
3,POL-002,Política de Férias,Direito a férias,vigente,O colaborador tem direito a 30 dias de férias após completar 12 meses de trabalho.
4,POL-002,Política de Férias,Fracionamento,vigente,As férias podem ser divididas em até 3 períodos. Um dos períodos deve ter no mínimo 14 dias corridos. Os demais períodos devem ter no mínimo 5 dias corridos cada.
5,POL-002,Política de Férias,Como solicitar,vigente,A solicitação é feita no portal de Pessoas e Cultura com antecedência mínima de 30 dias em relação ao início das férias. O gestor direto aprova ou recusa em até 5 dias úteis.
6,POL-002,Política de Férias,Venda de dias,vigente,O colaborador pode vender até 10 dias de férias. Esse pedido é feito junto com a solicitação de férias.
7,POL-003,Política de Benefícios,Vale-refeição,vigente,O valor do vale-refeição é de 40 reais por dia útil trabalhado. O crédito é feito no primeiro dia útil de cada mês.
8,POL-003,Política de Benefícios,Plano de saúde,vigente,O plano de saúde está disponível a partir do primeiro dia de trabalho. A inclusão de dependentes tem coparticipação de 30 por cento sobre o valor da mensalidade do dependente.
9,POL-003,Política de Benefícios,Vale-transporte,vigente,O vale-transporte é oferecido para os dias presenciais. O colaborador informa a rota no portal de Pessoas e Cultura.



## Evidência da P1

In [4]:
# Evidência da P1
print(f"Número total de chunks: {len(chunks_df)}")
print("\nNúmero de chunks por documento:")
print(chunks_df["doc_id"].value_counts().sort_index())

print("\nExemplo completo de chunk:")
exemplo = chunks_df.iloc[0].to_dict()
for chave, valor in exemplo.items():
    print(f"{chave}: {valor}")

Número total de chunks: 46

Número de chunks por documento:
doc_id
FAQ-001    6
POL-001    3
POL-002    4
POL-003    4
POL-004    3
POL-005    3
POL-006    4
POL-007    4
POL-008    4
POL-009    4
POL-010    3
POL-011    4
Name: count, dtype: int64

Exemplo completo de chunk:
doc_id: POL-001
titulo do documento: Política de Onboarding
nome da seção: Objetivo
status: vigente
texto da seção: Esta política define as etapas dos primeiros 30 dias de um novo colaborador na Horizonte Tech.


In [5]:
# Calcula medidas simples para documentar a variação dos chunks.
tamanhos_chunks = chunks_df["texto da seção"].apply(
    lambda texto: len(re.findall(r"[^\W_]+(?:[-'][^\W_]+)*", texto, flags=re.UNICODE))
)

print(f"Quantidade total de chunks: {len(tamanhos_chunks)}")
print(f"Menor chunk em palavras: {tamanhos_chunks.min()}")
print(f"Maior chunk em palavras: {tamanhos_chunks.max()}")
print(f"Média de palavras por chunk: {tamanhos_chunks.mean():.0f}")

Quantidade total de chunks: 46
Menor chunk em palavras: 10
Maior chunk em palavras: 64
Média de palavras por chunk: 21


---

## P2 - Indexação com TF-IDF

In [6]:
# P2 - Indexação com TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer


def criar_indice_tfidf(textos):
    """Ajusta o índice TF-IDF e transforma todos os chunks em vetores."""
    # Mantém acentos e stopwords; somente lowercase=True é aplicado.
    vectorizer = TfidfVectorizer(lowercase=True)
    matriz_tfidf = vectorizer.fit_transform(textos)
    return vectorizer, matriz_tfidf


vectorizer, matriz_tfidf = criar_indice_tfidf(chunks_df["texto da seção"])

print(f"Shape da matriz TF-IDF: {matriz_tfidf.shape}")
print(f"Tamanho do vocabulário: {len(vectorizer.vocabulary_)}")
print(
    "Decisão utilizada foi usar apenas: lowercase=True; " \
    "O lowercase foi adotado para evitar que diferenças de capitalização gerassem termos distintos e fragmentassem desnecessariamente o vocabulário."
)

Shape da matriz TF-IDF: (46, 350)
Tamanho do vocabulário: 350
Decisão utilizada foi usar apenas: lowercase=True; O lowercase foi adotado para evitar que diferenças de capitalização gerassem termos distintos e fragmentassem desnecessariamente o vocabulário.


---

## P3 - Recuperação top-k com regra de vigência

In [7]:
from sklearn.metrics.pairwise import cosine_similarity


def buscar(pergunta, k=3, apenas_vigentes=True):
    """Retorna os k chunks mais similares, filtrando vigentes por padrão."""
    if k <= 0:
        return pd.DataFrame(columns=["doc_id", "seção", "status", "score"])

    indices_busca = chunks_df.index[chunks_df["status"].eq("vigente")] if apenas_vigentes else chunks_df.index
    matriz_vigente = matriz_tfidf.loc[indices_busca] if hasattr(matriz_tfidf, "loc") else matriz_tfidf[indices_busca]
    vetor_pergunta = vectorizer.transform([pergunta])
    scores = cosine_similarity(vetor_pergunta, matriz_vigente).ravel()

    resultados = chunks_df.loc[indices_busca].copy()
    resultados["score"] = scores
    resultados = resultados.sort_values(
        by="score", ascending=False, kind="mergesort"
    ).head(k)
    resultados = resultados.rename(columns={"nome da seção": "seção"})
    return resultados[
        ["doc_id", "seção", "status", "score", "titulo do documento", "texto da seção"]
    ].reset_index(drop=True)


perguntas_p3 = pd.read_csv(base_dir / "perguntas_gabarito.csv")
perguntas_p3 = perguntas_p3[perguntas_p3["pergunta_id"].isin(["P01", "P02", "P10"])]

for _, pergunta in perguntas_p3.iterrows():
    print(f"{pergunta['pergunta_id']}: {pergunta['pergunta']}")
    evidencia = buscar(pergunta["pergunta"], k=3).copy()
    evidencia["score"] = evidencia["score"].map(lambda score: f"{score:.2f}")
    display(evidencia[["doc_id", "seção", "status", "score"]])

perguntas_validacao_revogados = pd.read_csv(base_dir / "perguntas_gabarito.csv")
resultados_validacao_revogados = [
    buscar(pergunta["pergunta"], k=3)
    for _, pergunta in perguntas_validacao_revogados.iterrows()
]
quantidade_perguntas_verificadas = len(resultados_validacao_revogados)
quantidade_chunks_analisados = sum(
    len(resultado_busca) for resultado_busca in resultados_validacao_revogados
)
quantidade_chunks_revogados = sum(
    (resultado_busca["status"] == "revogada").sum()
    for resultado_busca in resultados_validacao_revogados
)

assert quantidade_chunks_revogados == 0
print(f"Perguntas verificadas: {quantidade_perguntas_verificadas}")
print(f"Chunks analisados: {quantidade_chunks_analisados}")
print(f"Chunks revogados encontrados: {quantidade_chunks_revogados}")
print("Validação concluída: nenhuma busca retornou chunk revogado.")

P01: Com quantos dias de antecedência devo solicitar minhas férias?


,doc_id,seção,status,score
0,POL-002,Como solicitar,vigente,0.41
1,POL-002,Venda de dias,vigente,0.41
2,POL-002,Direito a férias,vigente,0.25


P02: Quantos dias por semana posso trabalhar de forma remota?


,doc_id,seção,status,score
0,POL-005,Regra de trabalho remoto,vigente,0.53
1,FAQ-001,Posso trabalhar remoto todos os dias?,vigente,0.20
2,POL-008,Entrevista de saída,vigente,0.17


P10: Qual é a política de estacionamento da empresa?


,doc_id,seção,status,score
0,POL-003,Auxílio para trabalho remoto,vigente,0.27
1,POL-008,Aviso,vigente,0.21
2,POL-001,Objetivo,vigente,0.18


Perguntas verificadas: 10
Chunks analisados: 30
Chunks revogados encontrados: 0
Validação concluída: nenhuma busca retornou chunk revogado.


---

## P4 - Resposta extrativa e regra de não encontrado

In [8]:
# P4 - Resposta extrativa e regra de não encontrado
THRESHOLD = 0.30
MENSAGEM_NAO_ENCONTRADO = (
    "Não encontrei essa informação nas políticas vigentes. Procure a área de Pessoas e Cultura."
)


def titulo_markdown_nivel_1(doc_id, texto):
    """Obtém o título do primeiro H1, sem repetir o doc_id na fonte."""
    for linha in texto.splitlines():
        if linha.startswith("# "):
            titulo = linha[2:].strip()
            return re.sub(rf"^{re.escape(doc_id)}\s*", "", titulo)
    return ""


titulos_por_doc = {
    linha["doc_id"]: titulo_markdown_nivel_1(linha["doc_id"], linha["texto"])
    for _, linha in df.iterrows()
}


def fonte_do_chunk(vencedor):
    """Monta a fonte do chunk e rastreia referências explícitas do FAQ."""
    doc_id = vencedor["doc_id"]
    titulo = titulos_por_doc[doc_id]
    fonte = f"{doc_id} | {titulo} | Seção: {vencedor['seção']}"

    if doc_id == "FAQ-001":
        referencias = dict.fromkeys(
            re.findall(r"\bPOL-\d{3}\b", vencedor["texto da seção"])
        )
        origens = [
            f"Origem: {referencia} | {titulos_por_doc[referencia]}"
            for referencia in referencias
            if referencia in titulos_por_doc
        ]
        if origens:
            fonte = f"{fonte} | " + " | ".join(origens)

    return fonte


def montar_saida(pergunta, resposta, fonte, score, status):
    """Monta a resposta nos cinco campos exigidos pelo SPEC."""
    return (
        f"PERGUNTA: {pergunta}\n"
        f"RESPOSTA: {resposta}\n"
        f"FONTE: {fonte}\n"
        f"SCORE: {score:.2f}\n"
        f"STATUS: {status}"
    )


def responder(pergunta):
    """Busca o melhor chunk e retorna a resposta no formato exigido."""
    vencedor = buscar(pergunta, k=1, apenas_vigentes=True).iloc[0]
    score = float(vencedor["score"])

    # O threshold usa o score real; o arredondamento ocorre apenas na saída.
    if score < THRESHOLD:
        return montar_saida(
            pergunta,
            MENSAGEM_NAO_ENCONTRADO,
            "nenhuma",
            score,
            "nao_encontrado",
        )

    return montar_saida(
        pergunta,
        vencedor["texto da seção"],
        fonte_do_chunk(vencedor),
        score,
        "encontrado",
    )


perguntas_gabarito = pd.read_csv(base_dir / "perguntas_gabarito.csv")
pergunta_p02 = perguntas_gabarito.loc[
    perguntas_gabarito["pergunta_id"].eq("P02"), "pergunta"
].iloc[0]
pergunta_p10 = perguntas_gabarito.loc[
    perguntas_gabarito["pergunta_id"].eq("P10"), "pergunta"
].iloc[0]


print(f"Threshold: {THRESHOLD:.2f}")
print("\nSaída P02:")
print(responder(pergunta_p02))
print("\nSaída P10:")
print(responder(pergunta_p10))

Threshold: 0.30

Saída P02:
PERGUNTA: Quantos dias por semana posso trabalhar de forma remota?
RESPOSTA: O colaborador pode trabalhar de forma remota em até 3 dias por semana. Os dias presenciais obrigatórios são terça-feira e quinta-feira.
FONTE: POL-005 | Política de Trabalho Híbrido (versão 2) | Seção: Regra de trabalho remoto
SCORE: 0.53
STATUS: encontrado

Saída P10:
PERGUNTA: Qual é a política de estacionamento da empresa?
RESPOSTA: Não encontrei essa informação nas políticas vigentes. Procure a área de Pessoas e Cultura.
FONTE: nenhuma
SCORE: 0.27
STATUS: nao_encontrado


In [9]:
# Evidência P4/P5 - P02 com e sem filtro de vigência
resultado_p02_apenas_vigentes = buscar(
    pergunta_p02, k=1, apenas_vigentes=True
).iloc[0]
resultado_p02_todos_status = buscar(
    pergunta_p02, k=1, apenas_vigentes=False
).iloc[0]

print("Cenário 1 — apenas_vigentes=True:")
print(
    f"doc_id: {resultado_p02_apenas_vigentes['doc_id']} | "
    f"seção: {resultado_p02_apenas_vigentes['seção']} | "
    f"status: {resultado_p02_apenas_vigentes['status']} | "
    f"score: {resultado_p02_apenas_vigentes['score']:.4f}"
)

print("Cenário 2 — apenas_vigentes=False:")
print(
    f"doc_id: {resultado_p02_todos_status['doc_id']} | "
    f"seção: {resultado_p02_todos_status['seção']} | "
    f"status: {resultado_p02_todos_status['status']} | "
    f"score: {resultado_p02_todos_status['score']:.4f}"
)

assert resultado_p02_apenas_vigentes["doc_id"] == "POL-005"
assert resultado_p02_apenas_vigentes["status"] == "vigente"
assert resultado_p02_todos_status["doc_id"] == "POL-004"
assert resultado_p02_todos_status["status"] == "revogada"
assert resultado_p02_todos_status["score"] > resultado_p02_apenas_vigentes["score"]

print("Validação concluída: sem o filtro, a POL-004 revogada vence a POL-005 vigente.")

Cenário 1 — apenas_vigentes=True:
doc_id: POL-005 | seção: Regra de trabalho remoto | status: vigente | score: 0.5276
Cenário 2 — apenas_vigentes=False:
doc_id: POL-004 | seção: Regra de trabalho remoto | status: revogada | score: 0.6571
Validação concluída: sem o filtro, a POL-004 revogada vence a POL-005 vigente.


**Justificativa da escolha do Threshold: 0.30**:

THRESHOLD = 0.30, definido com base nos scores observados: a consulta não encontrada P10 obteve 0.27, enquanto a menor pontuação entre as consultas encontradas foi 0.32 (P09). O valor permite separar esses casos no conjunto de avaliação utilizado.

### Validação da rastreabilidade do FAQ-001

Este teste confirma que, quando o FAQ-001 é recuperado, a fonte preserva o FAQ e a política de origem explicitamente mencionada.


In [10]:
faq_chunks_teste = chunks_df[
    chunks_df["doc_id"].eq("FAQ-001")
    & chunks_df["texto da seção"].str.contains(r"\bPOL-\d{3}\b", regex=True)
]
faq_chunk_teste = faq_chunks_teste.iloc[0]
pergunta_faq_teste = faq_chunk_teste["texto da seção"]
resultado_faq_p4 = buscar(pergunta_faq_teste, k=1).iloc[0]
referencia_faq_teste = re.search(
    r"\bPOL-\d{3}\b", faq_chunk_teste["texto da seção"]
).group()
fonte_faq_p4 = fonte_do_chunk(resultado_faq_p4)
resposta_faq_p4 = responder(pergunta_faq_teste)

assert resultado_faq_p4["doc_id"] == "FAQ-001"
assert f"FAQ-001 | {titulos_por_doc['FAQ-001']}" in fonte_faq_p4
assert f"Origem: {referencia_faq_teste} | {titulos_por_doc[referencia_faq_teste]}" in fonte_faq_p4
assert resposta_faq_p4 == montar_saida(
    pergunta_faq_teste,
    resultado_faq_p4["texto da seção"],
    fonte_faq_p4,
    float(resultado_faq_p4["score"]),
    "encontrado",
)

print("Validação do FAQ-001 concluída:", fonte_faq_p4)

Validação do FAQ-001 concluída: FAQ-001 | Perguntas Frequentes de Novos Colaboradores | Seção: Quando recebo meus acessos? | Origem: POL-001 | Política de Onboarding


---

## P5 Avaliação com o gabarito

In [11]:
# P5 - Avaliação com o gabarito


def avaliar_perguntas(perguntas):
    """Executa buscar e responder para cada pergunta do gabarito."""
    resultados = []

    for _, pergunta_gabarito in perguntas.iterrows():
        pergunta_texto = pergunta_gabarito["pergunta"]
        resultados_busca = buscar(pergunta_texto, k=3)
        resposta = responder(pergunta_texto)
        status = resposta.rsplit("STATUS: ", 1)[1].strip()
        score = float(resultados_busca.iloc[0]["score"])
        doc_retornado_1 = resultados_busca.iloc[0]["doc_id"]
        doc_esperado = pergunta_gabarito["doc_esperado"]

        if doc_esperado == "nao_encontrado":
            # P10 é verificada separadamente e não entra no Hit@1/Hit@3.
            hit1 = None
            hit3 = None
            acerto = "sim" if status == "nao_encontrado" else "não"
        else:
            hit1 = int(doc_retornado_1 == doc_esperado)
            hit3 = int(doc_esperado in resultados_busca["doc_id"].tolist())
            acerto = "sim" if hit1 else "não"

        resultados.append(
            {
                "pergunta_id": pergunta_gabarito["pergunta_id"],
                "doc_esperado": doc_esperado,
                "doc_retornado_1": doc_retornado_1,
                "score": score,
                "STATUS": status,
                "acerto": acerto,
                "hit1": hit1,
                "hit3": hit3,
            }
        )

    return pd.DataFrame(resultados)


def calcular_metricas(avaliacao):
    """Calcula Hit@1 e Hit@3 somente nas perguntas com documento esperado."""
    perguntas_com_resposta = avaliacao[avaliacao["doc_esperado"] != "nao_encontrado"]
    hit1 = perguntas_com_resposta["hit1"].mean()
    hit3 = perguntas_com_resposta["hit3"].mean()
    return hit1, hit3


def verificar_p10(avaliacao):
    """Retorna o resultado da verificação separada da P10."""
    return avaliacao.loc[avaliacao["pergunta_id"].eq("P10"), "acerto"].iloc[0]


perguntas_gabarito_p5 = pd.read_csv(base_dir / "perguntas_gabarito.csv")
avaliacao_p5 = avaliar_perguntas(perguntas_gabarito_p5)
hit1_p5, hit3_p5 = calcular_metricas(avaliacao_p5)
resultado_p10_p5 = verificar_p10(avaliacao_p5)

scores_positivos = avaliacao_p5.loc[
    avaliacao_p5["pergunta_id"].isin([f"P{i:02d}" for i in range(1, 10)]), "score"
]
scores_negativos = avaliacao_p5.loc[
    avaliacao_p5["pergunta_id"].eq("P10"), "score"
]
menor_score_positivo = scores_positivos.min()
maior_score_negativo = scores_negativos.max()
folga_scores = menor_score_positivo - maior_score_negativo

avaliacao_p5_apresentacao = avaliacao_p5[
    ["pergunta_id", "doc_esperado", "doc_retornado_1", "score", "STATUS", "acerto"]
].copy()
avaliacao_p5_apresentacao["score"] = avaliacao_p5_apresentacao["score"].map(
    lambda score: f"{score:.4f}"
)

print("Tabela das 10 perguntas:")
display(avaliacao_p5_apresentacao)
print(f"Menor score positivo: {menor_score_positivo:.4f}")
print(f"Maior score negativo: {maior_score_negativo:.4f}")
print(f"Folga: {folga_scores:.4f}")
print(f"Hit@1: {hit1_p5:.2f}")
print(f"Hit@3: {hit3_p5:.2f}")
print(f"Verificação da P10: {resultado_p10_p5}")

p04_resultado = avaliacao_p5.loc[avaliacao_p5["pergunta_id"].eq("P04")].iloc[0]
p04_score = f"{p04_resultado['score']:.4f}".replace(".", ",")

Tabela das 10 perguntas:


,pergunta_id,doc_esperado,doc_retornado_1,score,STATUS,acerto
0,P01,POL-002,POL-002,0.4107,encontrado,sim
1,P02,POL-005,POL-005,0.5276,encontrado,sim
2,P03,POL-003,POL-003,0.5589,encontrado,sim
3,P04,POL-006,POL-006,0.3080,encontrado,sim
4,P05,POL-009,POL-009,0.3306,encontrado,sim
5,P06,POL-007,POL-007,0.4958,encontrado,sim
6,P07,POL-008,POL-008,0.3883,encontrado,sim
7,P08,POL-009,POL-009,0.4686,encontrado,sim
8,P09,POL-010,POL-010,0.3228,encontrado,sim
9,P10,nao_encontrado,POL-003,0.2661,nao_encontrado,sim


Menor score positivo: 0.3080
Maior score negativo: 0.2661
Folga: 0.0419
Hit@1: 1.00
Hit@3: 1.00
Verificação da P10: sim


In [12]:
# Evidência P5 - Saída completa e validação da resposta extrativa


def extrair_campo_saida(saida, campo):
    """Extrai um campo da saída textual de responder sem alterar seu conteúdo."""
    prefixo = f"{campo}: "
    return next(linha[len(prefixo):] for linha in saida.splitlines() if linha.startswith(prefixo))


for _, pergunta_gabarito in perguntas_gabarito_p5.iterrows():
    pergunta_id = pergunta_gabarito["pergunta_id"]
    pergunta_texto = pergunta_gabarito["pergunta"]
    saida_responder = responder(pergunta_texto)
    resultado_busca = buscar(pergunta_texto, k=1)
    status = extrair_campo_saida(saida_responder, "STATUS")

    print(f"{pergunta_id}")
    print(saida_responder)
    print()

    if status == "encontrado":
        chunk_vencedor = resultado_busca.iloc[0]["texto da seção"]
        resposta = extrair_campo_saida(saida_responder, "RESPOSTA")
        assert resposta == chunk_vencedor
    else:
        assert status == "nao_encontrado"
        assert pergunta_id == "P10"
        assert extrair_campo_saida(saida_responder, "FONTE") == "nenhuma"

print("Validação da resposta extrativa concluída para P01–P09.")
print("Validação de P10 concluída: STATUS nao_encontrado.")

P01
PERGUNTA: Com quantos dias de antecedência devo solicitar minhas férias?
RESPOSTA: A solicitação é feita no portal de Pessoas e Cultura com antecedência mínima de 30 dias em relação ao início das férias. O gestor direto aprova ou recusa em até 5 dias úteis.
FONTE: POL-002 | Política de Férias | Seção: Como solicitar
SCORE: 0.41
STATUS: encontrado

P02
PERGUNTA: Quantos dias por semana posso trabalhar de forma remota?
RESPOSTA: O colaborador pode trabalhar de forma remota em até 3 dias por semana. Os dias presenciais obrigatórios são terça-feira e quinta-feira.
FONTE: POL-005 | Política de Trabalho Híbrido (versão 2) | Seção: Regra de trabalho remoto
SCORE: 0.53
STATUS: encontrado

P03
PERGUNTA: Qual é o valor do vale-refeição por dia?
RESPOSTA: O valor do vale-refeição é de 40 reais por dia útil trabalhado. O crédito é feito no primeiro dia útil de cada mês.
FONTE: POL-003 | Política de Benefícios | Seção: Vale-refeição
SCORE: 0.56
STATUS: encontrado

P04
PERGUNTA: Qual é o prazo p

In [13]:
# Evidência P5 - TOP-3 da recuperação da P04
pergunta_p04 = perguntas_gabarito_p5.loc[
    perguntas_gabarito_p5["pergunta_id"].eq("P04"), "pergunta"
].iloc[0]
top3_p04 = buscar(pergunta_p04, k=3).copy()
top3_p04.insert(0, "posição", range(1, len(top3_p04) + 1))

print("P04 — TOP-3 da recuperação:")
for _, resultado_p04 in top3_p04.iterrows():
    print(
        f"{resultado_p04['posição']}. {resultado_p04['doc_id']} | "
        f"{resultado_p04['seção']} | score: {resultado_p04['score']:.4f}"
    )

P04 — TOP-3 da recuperação:
1. POL-006 | Prazo para envio | score: 0.3080
2. POL-001 | Primeiro mês | score: 0.1676
3. POL-006 | Pagamento | score: 0.1426


### Análise de near-threshold

P04 é um caso near-threshold: a POL-006 foi recuperada corretamente em primeiro lugar, mas com score próximo de 0,30.  
As duas alternativas do TOP-3 apresentam scores bem menores, mantendo a recuperação correta.  
A causa provável é a baixa sobreposição lexical entre a pergunta e o texto da política na representação TF-IDF.  
Como tentativa de correção, seria avaliado o uso de n-gramas no TF-IDF para capturar combinações de termos presentes na pergunta.  

---

### *Experimentações*:

In [16]:
# Experimento separado - mini-ablacao de pre-processamento TF-IDF
from sklearn.metrics.pairwise import cosine_similarity


stopwords_pt_ablation = [
    "a", "à", "às", "ao", "aos", "as", "com", "como", "da", "das", "de", "do", "dos",
    "e", "é", "em", "entre", "essa", "essas", "esse", "esses", "esta",
    "estas", "este", "estes", "eu", "foi", "foram", "há", "isso", "isto", "já",
    "mas", "me", "mesmo", "na", "nas", "não", "no", "nos", "num", "numa",
    "o", "os", "ou", "para", "pela", "pelas", "pelo", "pelos", "por",
    "qual", "quando", "que", "se", "sem", "ser", "são", "também", "tem",
    "tendo", "tenho", "ter", "um", "uma", "umas", "uns", "vai", "você",
]

coluna_texto_ablation = next(
    coluna for coluna in chunks_df.columns if coluna.startswith("texto da se")
)
textos_ablation = chunks_df[coluna_texto_ablation].tolist()
perguntas_ablation = perguntas_gabarito_p5.copy()
indices_vigentes_ablation = chunks_df.index[chunks_df["status"].eq("vigente")]


def avaliar_configuracao_ablation(nome, vectorizer_ablation):
    """Avalia uma configuracao local sem alterar o pipeline oficial."""
    matriz_ablation = vectorizer_ablation.fit_transform(textos_ablation)
    resultados_ablation = []

    for _, pergunta_ablation in perguntas_ablation.iterrows():
        vetor_pergunta_ablation = vectorizer_ablation.transform([pergunta_ablation["pergunta"]])
        matriz_vigente_ablation = matriz_ablation[indices_vigentes_ablation]
        scores_ablation = cosine_similarity(
            vetor_pergunta_ablation, matriz_vigente_ablation
        ).ravel()
        ranking_ablation = chunks_df.loc[indices_vigentes_ablation].copy()
        ranking_ablation["score"] = scores_ablation
        ranking_ablation = ranking_ablation.sort_values("score", ascending=False).head(3)

        doc_esperado_ablation = pergunta_ablation["doc_esperado"]
        if doc_esperado_ablation == "nao_encontrado":
            hit1_ablation = None
            hit3_ablation = None
        else:
            hit1_ablation = int(ranking_ablation.iloc[0]["doc_id"] == doc_esperado_ablation)
            hit3_ablation = int(doc_esperado_ablation in ranking_ablation["doc_id"].tolist())

        resultados_ablation.append(
            {
                "pergunta_id": pergunta_ablation["pergunta_id"],
                "doc_esperado": doc_esperado_ablation,
                "score": float(ranking_ablation.iloc[0]["score"]),
                "hit1": hit1_ablation,
                "hit3": hit3_ablation,
            }
        )

    resultados_ablation = pd.DataFrame(resultados_ablation)
    positivos_ablation = resultados_ablation[
        resultados_ablation["doc_esperado"] != "nao_encontrado"
    ]
    score_p10_ablation = resultados_ablation.loc[
        resultados_ablation["pergunta_id"].eq("P10"), "score"
    ].iloc[0]
    menor_positivo_ablation = positivos_ablation["score"].min()
    return {
        "configuracao": nome,
        "Hit@1": positivos_ablation["hit1"].mean(),
        "Hit@3": positivos_ablation["hit3"].mean(),
        "menor_score_positivo": menor_positivo_ablation,
        "score_p10": score_p10_ablation,
        "folga": menor_positivo_ablation - score_p10_ablation,
    }


configuracoes_ablation = [
    ("BASE", TfidfVectorizer(lowercase=True)),
    ("SEM STOPWORDS", TfidfVectorizer(lowercase=True, stop_words=stopwords_pt_ablation)),
    ("SEM ACENTOS", TfidfVectorizer(lowercase=True, strip_accents="unicode")),
]

resultados_ablation = [
    avaliar_configuracao_ablation(nome, vectorizer_ablation)
    for nome, vectorizer_ablation in configuracoes_ablation
]

tabela_ablation = pd.DataFrame(resultados_ablation)
tabela_ablation_apresentacao = tabela_ablation.copy()
for coluna_ablation in [
    "Hit@1", "Hit@3", "menor_score_positivo", "score_p10", "folga"
]:
    tabela_ablation_apresentacao[coluna_ablation] = tabela_ablation_apresentacao[
        coluna_ablation
    ].map(lambda valor_ablation: f"{valor_ablation:.4f}")

tabela_ablation_apresentacao = tabela_ablation_apresentacao.rename(
    columns={
        "configuracao": "configuracao",
        "menor_score_positivo": "menor score positivo",
        "score_p10": "score P10",
    }
)
print("Mini-ablacao de pre-processamento - mesmo corpus, perguntas e chunks vigentes:")
display(
    tabela_ablation_apresentacao[
        ["configuracao", "Hit@1", "Hit@3", "menor score positivo", "score P10", "folga"]
    ]
)

resultado_base_ablation = resultados_ablation[0]
print("Analise:")
for resultado_ablation in resultados_ablation:
    diferenca_folga_ablation = resultado_ablation["folga"] - resultado_base_ablation["folga"]
    if diferenca_folga_ablation > 0:
        direcao_folga_ablation = "aumentou"
    elif diferenca_folga_ablation < 0:
        direcao_folga_ablation = "diminuiu"
    else:
        direcao_folga_ablation = "permaneceu igual"
    mudou_hit_ablation = (
        resultado_ablation["Hit@1"] != resultado_base_ablation["Hit@1"]
        or resultado_ablation["Hit@3"] != resultado_base_ablation["Hit@3"]
    )
    print(
        f"{resultado_ablation['configuracao']}: a folga {direcao_folga_ablation} "
        f"para {resultado_ablation['folga']:.4f}; Hit@1/Hit@3 "
        f"{'mudaram' if mudou_hit_ablation else 'não mudaram'} em relacao a BASE."
    )


assert matriz_tfidf.shape == (46, 350)
assert vectorizer is not None


Mini-ablacao de pre-processamento - mesmo corpus, perguntas e chunks vigentes:


,configuracao,Hit@1,Hit@3,menor score positivo,score P10,folga
0,BASE,1.0000,1.0000,0.3080,0.2661,0.0419
1,SEM STOPWORDS,1.0000,1.0000,0.3776,0.2242,0.1534
2,SEM ACENTOS,1.0000,1.0000,0.3080,0.2661,0.0419


Analise:
BASE: a folga permaneceu igual para 0.0419; Hit@1/Hit@3 não mudaram em relacao a BASE.
SEM STOPWORDS: a folga aumentou para 0.1534; Hit@1/Hit@3 não mudaram em relacao a BASE.
SEM ACENTOS: a folga permaneceu igual para 0.0419; Hit@1/Hit@3 não mudaram em relacao a BASE.


In [15]:
# Avaliacao complementar - conferencia das secoes P01-P09
perguntas_secao = perguntas_gabarito_p5[
    perguntas_gabarito_p5["pergunta_id"].isin([f"P{i:02d}" for i in range(1, 10)])
].copy()
coluna_secao_retornada = "\u0073\u0065\u00e7\u00e3\u006f"

resultados_secao = []
for _, pergunta_secao in perguntas_secao.iterrows():
    resultado_secao = buscar(pergunta_secao["pergunta"], k=3).iloc[0]
    doc_id_retornado_secao = resultado_secao["doc_id"]
    secao_retornada = resultado_secao[coluna_secao_retornada]
    resultados_secao.append(
        {
            "pergunta_id": pergunta_secao["pergunta_id"],
            "doc_id esperado": pergunta_secao["doc_esperado"],
            "secao esperada": pergunta_secao["secao_esperada"],
            "doc_id retornado": doc_id_retornado_secao,
            "secao retornada": secao_retornada,
            "score": float(resultado_secao["score"]),
            "documento correto": doc_id_retornado_secao == pergunta_secao["doc_esperado"],
            "secao correta": secao_retornada == pergunta_secao["secao_esperada"],
        }
    )

avaliacao_secoes_p01_p09 = pd.DataFrame(resultados_secao)
avaliacao_secoes_apresentacao = avaliacao_secoes_p01_p09.copy()
avaliacao_secoes_apresentacao["score"] = avaliacao_secoes_apresentacao["score"].map(
    lambda score_secao: f"{score_secao:.4f}"
)

print("Avaliacao complementar das secoes P01-P09:")
display(avaliacao_secoes_apresentacao)


Avaliacao complementar das secoes P01-P09:


,pergunta_id,doc_id esperado,secao esperada,doc_id retornado,secao retornada,score,documento correto,secao correta
0,P01,POL-002,Como solicitar,POL-002,Como solicitar,0.4107,True,True
1,P02,POL-005,Regra de trabalho remoto,POL-005,Regra de trabalho remoto,0.5276,True,True
2,P03,POL-003,Vale-refeição,POL-003,Vale-refeição,0.5589,True,True
3,P04,POL-006,Prazo para envio,POL-006,Prazo para envio,0.3080,True,True
4,P05,POL-009,Senhas,POL-009,Senhas,0.3306,True,True
5,P06,POL-007,Ciclo,POL-007,Ciclo,0.4958,True,True
6,P07,POL-008,Devolução de equipamentos,POL-008,Devolução de equipamentos,0.3883,True,True
7,P08,POL-009,Uso de ferramentas de inteligência artificial externas,POL-009,Uso de ferramentas de inteligência artificial externas,0.4686,True,True
8,P09,POL-010,Orçamento,POL-010,Orçamento,0.3228,True,True
